# 📓 03 — 历史 pattern 匹配 + 自定义 forward return

**目标**: 学会用 `find_similar_patterns()` 找历史相似 K 线,改参数看不同结果。

**适合**: 想知道"当前形态历史上后续 5/20/60 日怎么走"的用户。

---

**默认参数**: 20d pattern length, top 10 matches, 5d forward。
本 notebook 演示 3 种变体:
1. **短 pattern (10d) + 长 forward (60d)**: 看"最近 2 周形态"后续 3 个月
2. **长 pattern (60d) + 短 forward (5d)**: 看"最近 3 个月形态"后续 1 周
3. **更严格匹配 (top 5)**: 只看最像的 5 个

In [ ]:
# 第 0 步: import (含 robust path 修复)
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    for cand in [cwd, *cwd.parents]:
        if (cand / 'src').is_dir() and (cand / 'config').is_dir():
            return cand
    return cwd

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.patterns import find_similar_patterns

In [ ]:
# 第 1 步: QQQ 默认 20d pattern × 5d forward
r_default = find_similar_patterns('QQQ', pattern_length=20, n_matches=10, forecast_horizon=5)
print("=== QQQ 默认 (20d × 5d fwd × top 10) ===")
for k, v in r_default.items():
    if isinstance(v, (int, float, str)):
        print(f"  {k:25s} {v}")

In [ ]:
# 第 2 步: 短 pattern 长 forward (10d × 60d fwd × top 10)
r_short = find_similar_patterns('QQQ', pattern_length=10, n_matches=10, forecast_horizon=60)
print("=== QQQ 短 pattern 长 forward (10d × 60d) ===")
print(f"  avg forward:    {r_short['avg_forward_return']:+.2f}%")
print(f"  win rate:       {r_short['win_rate']:.0%}")
print(f"  max forward:    {r_short['max_forward']:+.2f}%")
print(f"  min forward:    {r_short['min_forward']:+.2f}%")

In [ ]:
# 第 3 步: 4 指数 × 3 种 pattern 配置 对比
configs = [
    ("短 10d × 5d",  dict(pattern_length=10, n_matches=10, forecast_horizon=5)),
    ("中 20d × 5d",  dict(pattern_length=20, n_matches=10, forecast_horizon=5)),
    ("长 60d × 20d", dict(pattern_length=60, n_matches=10, forecast_horizon=20)),
]
print(f"{'config':18s} {'symbol':6s} {'avg':>7s} {'win':>6s} {'max':>7s} {'min':>7s}")
print("-" * 60)
for name, cfg in configs:
    for sym in ['DIA', 'QQQ', 'RSP', 'QQQE']:
        r = find_similar_patterns(sym, **cfg)
        print(f"{name:18s} {sym:6s} {r['avg_forward_return']:>+6.2f}% "
              f"{r['win_rate']:>5.0%} {r['max_forward']:>+6.2f}% {r['min_forward']:>+6.2f}%")
    print()

In [ ]:
# 第 4 步: 严格匹配 (top 5) 看最像的
r_top5 = find_similar_patterns('QQQ', pattern_length=20, n_matches=5, forecast_horizon=5)
print("=== QQQ 严格匹配 (top 5) ===")
print(f"  avg forward:    {r_top5['avg_forward_return']:+.2f}%")
print(f"  win rate:       {r_top5['win_rate']:.0%}")
print(f"\n  top_matches: ")
for i, m in enumerate(r_top5['top_matches'][:5], 1):
    print(f"    {i}. start {m['start_date']}, "
          f"corr {m['correlation']:.3f}, "
          f"fwd {m['forward_return']*100:+.2f}%")

## 🎯 练习

1. **找矛盾配置**: 哪种配置下 4 指数的 win rate 差距最大?最大多少?
2. **layer 切换**: 改 `'indices'` 到 `'sectors'`,看 11 GICS 行业的 pattern 匹配
3. **forecast_horizon = 1**: 看明天涨的概率,跟 5d 差多少?
4. **threshold 过滤**: 只看相关性 > 0.9 的 matches,样本少但更准